# Grid Load Management — Full MILP sensitivity of local ramping weight (CBC)

**This notebook is a runner, NOT a completed computational result.**

- Original frozen `multiwindow_inputs_frozen.npz`, 12 windows (W00–W11), 59 scenarios in total.
- Same fairness constraint (epsilon = 0.1), CLPU kernel (0.30, 0.15, 0.05), and supply-proxy and caps.
- Reoptimize binary switching/restoration decisions at each weight using the **unchanged v4 model**.
- Weight multipliers: 0, 0.1, 0.25, 0.5, 1, 2 of baseline 0.01.
- **Results are valid only after CBC execution, and solver termination/gap must be disclosed.**
- W00 is a 4-scenario control. Main results average **W01–W11** (5 scenarios each).

Upload `GRID_LOAD_MILP_PuLP_3_3_2_FIXED.zip` produced alongside this notebook.

In [ ]:
# Keep PuLP 3.3.2: the original frozen Grid Load Management MILP uses its 3.x modelling API.
%pip -q install "pulp==3.3.2" numpy pandas matplotlib
print("If you previously imported PuLP 4.x, select Runtime > Restart session now, then rerun from this cell.")


In [ ]:
import pulp, subprocess
print("PuLP:", pulp.__version__)
assert pulp.__version__ == "3.3.2", "Restart runtime after downgrading from PuLP 4.x"
assert hasattr(pulp, "LpVariable") and hasattr(pulp, "PULP_CBC_CMD")
def available_cbc():
    for api in ("PULP_CBC_CMD", "COIN_CMD"):
        try:
            solver = getattr(pulp, api)(msg=False)
            path = solver.available()
            if path:
                print("CBC available:", api, "at", path)
                return True
        except Exception as exc:
            print(api, "not available:", exc)
    return False
if not available_cbc():
    print("Installing system CBC... ")
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "coinor-cbc"], check=True)
assert available_cbc(), "CBC executable unavailable after installation"


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, subprocess, sys
uploaded=files.upload()
archives=[k for k in uploaded if k.lower().endswith('.zip')]
assert len(archives)==1, 'Upload only the provided GRID_LOAD_MILP_PuLP_3_3_2_FIXED.zip'
BASE=Path('/content/grid-load-management_cbc_sensitivity'); BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    assert all(not name.startswith('/') and '..' not in Path(name).parts for name in z.namelist())
    z.extractall(BASE)
assert (BASE/'run_full_cbc.py').is_file() and (BASE/'multiwindow_inputs_frozen.npz').is_file()
print('Unpacked:', BASE)
subprocess.run([sys.executable,'test_frozen_protocol.py'],cwd=BASE,check=True)
subprocess.run([sys.executable,'run_full_cbc.py','--dry-run'],cwd=BASE,check=True)

## Save results safely in your Google Drive

Mounting Drive requires your consent in Colab. Reusing the same result directory means already completed cases are skipped on subsequent sessions (assuming frozen inputs, solver settings, and model code have not changed).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/GRID_LOAD_MILP_FULL_MILP_RESULTS')
OUT.mkdir(parents=True,exist_ok=True)
print('Result storage:', OUT)

## First, limited smoke test (W00, multipliers 0 and 1)

This actually runs two full CBC MILPs. The original model is not altered. Review log status before running all cases.

In [ ]:
subprocess.run([sys.executable,'run_full_cbc.py',
    '--windows','0','--multipliers','0,1',
    '--time-limit','300','--gap','0.001','--output-dir',str(OUT)],
    cwd=BASE,check=True)
subprocess.run([sys.executable,'make_report.py','--results-dir',str(OUT)],cwd=BASE,check=True)

## Full run: 12 × 6 = 72 MILP instances

**Note:** Results are checkpointed case by case on Drive. Rerun this cell after an interruption to skip completed cases. Do not change the frozen file or solver arguments while resuming. Time-limited incumbents remain separately labeled.

In [ ]:
subprocess.run([sys.executable,'run_full_cbc.py',
    '--windows','all','--multipliers','all',
    '--time-limit','300','--gap','0.001','--output-dir',str(OUT)],
    cwd=BASE,check=True)
subprocess.run([sys.executable,'make_report.py','--results-dir',str(OUT)],cwd=BASE,check=True)

In [ ]:
import pandas as pd
from IPython.display import display
cases=pd.read_csv(OUT/'case_metrics.csv')
summary=pd.read_csv(OUT/'primary_11_windows_summary.csv')
display(summary)
print((OUT/'REPORT.md').read_text())
print('Feasibility-checked full-MILP cases:',cases['usable_feasible'].sum(), '/',72)
print('W07 v4-like baseline:',cases.loc[(cases.window_id==7)&(cases.lambda_multiplier==1),['cbc_termination','cbc_reported_relative_gap','validation','elapsed_s']])

In [ ]:
archive=shutil.make_archive('/content/GRID_LOAD_MILP_FULL_MILP_RESULTS','zip',root_dir=str(OUT))
files.download(archive)